In [1]:
import os 
import sys
import json
import random
from pathlib import Path
from dotenv import load_dotenv

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))

load_dotenv(project_root / ".env")


openrouter_key = os.getenv("OPENROUTER_API_KEY")
openai_key = os.getenv("OPENAI_API_KEY")

if not openrouter_key and not openai_key:
    raise EnvironmentError(
        "❌ No API key found!\n"
        "   Add OPENROUTER_API_KEY (recommended) or OPENAI_API_KEY to .env"
    )

from infrastructure.config import (
    CRAWL_OUT_DIR, VECTOR_DIR, EMBEDDING_MODEL, PROVIDER
)

random.seed(42)

provider = "openrouter" if openrouter_key else "OpenAI"
print("✅ Environment loaded")
print(f"🌐 Provider: {provider}")
print(f"📁 Project root: {project_root}")

✅ Environment loaded
🌐 Provider: openrouter
📁 Project root: d:\ai\agentic ai\ceysaid_intelligence_platform


In [2]:
from infrastructure.config import (
    validate, dump, CRAWL_OUT_DIR, MARKDOWN_DIR
)

try:
    validate()
    dump()
except Exception as e:
    print(f"⚠️  Config note: {e}")

print(f"\n✅ Output directories ready:")
print(f"   - Markdown: {MARKDOWN_DIR}")
print(f"   - JSONL: {CRAWL_OUT_DIR}")

2026-03-08 02:23:49.342 | INFO     | infrastructure.config:dump:340 - 
2026-03-08 02:23:49.343 | INFO     | infrastructure.config:dump:341 - CONFIGURATION (NON-SECRETS ONLY)
2026-03-08 02:23:49.344 | INFO     | infrastructure.config:dump:342 - ============================================================
2026-03-08 02:23:49.345 | INFO     | infrastructure.config:dump:344 - 
🌐 Provider:
2026-03-08 02:23:49.345 | INFO     | infrastructure.config:dump:345 -    Provider: openrouter
2026-03-08 02:23:49.345 | INFO     | infrastructure.config:dump:346 -    Model Tier: general
2026-03-08 02:23:49.346 | INFO     | infrastructure.config:dump:347 -    Chat Model: google/gemini-2.5-flash
2026-03-08 02:23:49.346 | INFO     | infrastructure.config:dump:348 -    Embedding Model: openai/text-embedding-3-small
2026-03-08 02:23:49.347 | INFO     | infrastructure.config:dump:349 -    Embedding Dimensions: 1536
2026-03-08 02:23:49.348 | INFO     | infrastructure.config:dump:351 - 
📁 Directories & Storage:



✅ Output directories ready:
   - Markdown: d:\ai\agentic ai\ceysaid_intelligence_platform\data\ceysaid_markdown
   - JSONL: d:\ai\agentic ai\ceysaid_intelligence_platform\data


In [3]:
jsonl_path = CRAWL_OUT_DIR / "ceysaid_corpus.jsonl"

if not jsonl_path.exists():
    raise FileExistsError(f"❌ Corpus not found.")

with open(jsonl_path, 'r', encoding="utf-8") as f:
    documents = [json.loads(line) for line in f]

print(f"✅ Loaded {len(documents)} documents")
print(f"📊 Total content size: {sum(len(d['content']) for d in documents):,} chars")

✅ Loaded 95 documents
📊 Total content size: 468,044 chars


## Applying chunking 

In [4]:
from applications.ingest_document_service.chunkers import parent_child_chunk

In [5]:
print("Running Parent child chunking")

child_chunks, parent_chunks = parent_child_chunk(documents=documents)

parent_path = CRAWL_OUT_DIR / "parent_chunk.jsonl"
child_path = CRAWL_OUT_DIR / "child_chunk.jsonl"

with open(parent_path, "w", encoding="utf-8") as f:
    for chunk in parent_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')

with open(child_path, "w", encoding="utf-8") as f:
    for chunk in child_chunks:
        f.write(json.dumps(chunk, ensure_ascii=False) + '\n')

print(f"✅ Parent child chunking complete: \nparent len : {len(parent_chunks)}\nchild len: {len(child_chunks)}")
print(f"💾 save to: \nparent path: {parent_path}\nchild chunks: {child_path}")

Running Parent child chunking
✅ Parent child chunking complete: 
parent len : 154
child len: 744
💾 save to: 
parent path: d:\ai\agentic ai\ceysaid_intelligence_platform\data\parent_chunk.jsonl
child chunks: d:\ai\agentic ai\ceysaid_intelligence_platform\data\child_chunk.jsonl


## Embeddings

In [6]:
from infrastructure.llm.embeddings import get_defailt_embedding

In [ ]:
from infrastructure.db.qdrant_db import QdrantStorage

emb = get_defailt_embedding()

db_client = QdrantStorage(embedding=emb)

print(f"\n🚀Creating qdrant storage")
# db_client.upsert_chunks(child_chunks)
print(f"✅ Vector store created!")


🚀Creating qdrant storage
✅ Vector store created!


In [8]:
test_query = "Required Documents for Vietnam Visa Application"

db_client.search_chunks(query=test_query)

[{'text': '# Vietnam Visa\n\n## **Required Documents for Vietnam Visa Application**\n\nTo apply for a Vietnam visa, applicants must provide the following documents:\n\n1. **White Background Photo**\n2. **Passport Copy**\n3. **Hotel Reservation**\n4. **Air Ticket**\n5. **Travel Itinerary**\n\n## **Urgent Visa Processing**\n\nFor travelers who need a **rush visa**, an urgent visa letter can be processed. After receiving the approved visa letter, travelers will need to pay the visa stamping fee and provide a visa photograph upon arrival at the airport in Vietnam.\n\n### **Important Notes:**\n\n* The visa fee is **non-refundable** under any circumstances.\n* Make sure all documents are accurate and complete to avoid processing delays.\n* Additional requirements may apply based on nationality and travel history.\n\nFor any further inquiries or assistance, feel free to contact our visa support team. Safe travels!',
  'url': 'https://ceysaid.com/vietnam-visa/',
  'title': 'Vietnam Visa - Ceys